In [5]:
%cd /home/asehgal/formulacode/datasmith

import json
import logging
import re
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from asv.commands.publish import Publish
from asv.config import Config
from asv.util import write_json

from datasmith import setup_environment
from datasmith.benchmark.collection import BenchmarkCollection
from datasmith.docker.context import ContextRegistry, Task
from datasmith.logging_config import configure_logging
from datasmith.scrape.scrape_dashboards import make_benchmark_from_html

configure_logging(level=logging.WARNING)
setup_environment()

/home/asehgal/formulacode/datasmith


In [8]:
context_registry_loc = Path("scratch/merged_context_registry_2025-09-06T06:14:21.165351.json")
benchmark_dir = Path("scratch/artifacts/pipeflush/benchmark_results")
results_dir = benchmark_dir / "results"
dashboard_dir = benchmark_dir / "dashboards"
context_registry = ContextRegistry.load_from_file(context_registry_loc)
assert results_dir.exists(), f"Results dir {results_dir.absolute()} does not exist"  # noqa: S101
dashboard_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
sha2tasks = {t.sha: t for t in context_registry.registry}


def get_files(pth: Path, pattern: str) -> dict[str, str]:
    return {str(f.relative_to(pth)): f.read_text(encoding="utf-8") for f in pth.rglob(pattern)}


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def merge_json_dict(dst_path: Path, payload: dict) -> None:
    """Merge dict payload into dst_path (if exists), else write it."""
    if dst_path.exists():
        try:
            existing = json.loads(dst_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            existing = {}
        if not isinstance(existing, dict):
            existing = {}
        existing.update(payload)
        dst_path.write_text(json.dumps(existing, indent=2, sort_keys=True), encoding="utf-8")
    else:
        ensure_dir(dst_path.parent)
        dst_path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def api_publish(repo_root: Path) -> None:
    """
    Publish results using ASV's Python API.
    - Reads repo_root/asv.conf.json
    - Forces results/html dirs to this repo_root
    - Creates Config and runs Publish.run(cfg)
    """
    asv_conf_path = repo_root / "asv.conf.json"
    conf_dict = json.loads(asv_conf_path.read_text(encoding="utf-8"))

    # Ensure minimal keys + local dirs
    conf_dict.setdefault("version", 1)
    conf_dict["results_dir"] = str((repo_root / "results").resolve())
    conf_dict["html_dir"] = str((repo_root / "html").resolve())
    conf_dict["repo_subdir"] = str((repo_root / "repo").resolve())
    conf_dict["project"] = str((repo_root / "project").resolve())

    # If repo is a local path, make it absolute; if URL, leave it alone
    repo_val = conf_dict.get("repo")
    if isinstance(repo_val, str) and not re.match(r"^https?://", repo_val):
        conf_dict["repo"] = str(Path(repo_val).resolve())

    cfg = Config.from_json(conf_dict)

    # Optional: write normalized config back to disk so it's inspectable/reproducible
    write_json(path=asv_conf_path, data=cfg.__dict__, api_version=1)

    # Publish!
    Publish.run(cfg)


benchmarked_imgs = list(results_dir.rglob("*pkg"))
dashboards = {}
with tempfile.TemporaryDirectory(prefix="asv-publish-") as tmpdirname:
    tmproot = Path(tmpdirname)
    repo_roots = {}
    repokey2task = {}
    for img in benchmarked_imgs:
        commit, tag = img.name.rsplit("-")[-2:]
        task = sha2tasks.get(commit)
        if not task:
            continue

        repo_key = f"{task.owner}/{task.repo}"
        repokey2task[repo_key] = Task(owner=task.owner, repo=task.repo, sha=None, tag=task.tag)
        contents = get_files(img, "*.json")
        needed = {"asv.conf.json", "benchmarks.json", "machine.json"}
        if not (needed.issubset({Path(p).name for p in contents}) and len(contents) >= 4):
            continue

        repo_root = repo_roots.setdefault(repo_key, tmproot / repo_key)
        results_out = repo_root / "results"
        html_out = repo_root / "html"
        ensure_dir(results_out)
        ensure_dir(html_out)

        asv_conf = json.loads(contents.pop(next(p for p in contents if p.endswith("asv.conf.json"))))
        benchmarks = json.loads(contents.pop(next(p for p in contents if p.endswith("benchmarks.json"))))
        machine = json.loads(contents.pop(next(p for p in contents if p.endswith("machine.json"))))

        merge_json_dict(results_out / "benchmarks.json", benchmarks)

        machine_name = machine.get("machine") or "imported"
        machine_dir = results_out / machine_name
        ensure_dir(machine_dir)

        # Save machine.json inside machine dir (ASV iterates machine.json files under results/) :contentReference[oaicite:3]{index=3}
        (machine_dir / "machine.json").write_text(json.dumps(machine, indent=2, sort_keys=True), encoding="utf-8")

        seen = set()
        for relpath, text in contents.items():
            if not relpath.endswith(".json"):
                continue
            base = Path(relpath).name
            if base in ("asv.conf.json", "benchmarks.json", "machine.json"):
                continue
            out = machine_dir / base
            if out.name in seen or out.exists():
                i = 1
                while True:
                    cand = machine_dir / f"{i}_{base}"
                    if not cand.exists():
                        out = cand
                        break
                    i += 1
            # write_text(out, text)
            out.write_text(text, encoding="utf-8")
            seen.add(out.name)

        # Patch & save config for this temp repo
        asv_conf = dict(asv_conf) if isinstance(asv_conf, dict) else {}
        asv_conf["results_dir"] = "results"
        asv_conf["html_dir"] = "html"
        # repo_url = parse_repo_url(asv_conf, repo_key)
        asv_conf["repo"] = f"https://github.com/{task.owner}/{task.repo}.git"
        # asv_conf.setdefault("project", "bencrepos/" + repo_key.replace("-", "/"))
        (repo_root / "asv.conf.json").write_text(json.dumps(asv_conf, indent=2, sort_keys=True), encoding="utf-8")

    # only keep the first 4 repo_roots items
    # only keep the first 4 repo_roots items

    failures = []
    dashboards = {}

    # Keep only the first 4 items
    def publish_one(repo_key, repo_root):
        task = repokey2task[repo_key]
        try:
            api_publish(repo_root)
            print(f"OK: {repo_root}")
            dashboard_collection = make_benchmark_from_html(
                base_url=f"{repo_root}/html",
                html_dir=f"{repo_root}/html",
                force=False,
            )
            if not dashboard_collection:
                print(f"No dashboard generated for: {task.owner}/{task.repo}")
                return (task, None, None)
            db_path = (dashboard_dir / f"{task.owner}_{task.repo}.fc.pkl").resolve()
            dashboard_collection.save(db_path)
            print(f"Dashboard saved to: {db_path}")
            return (task, dashboard_collection, None)  # noqa: TRY300
        except Exception as e:
            print(f"FAILED ({task.owner}/{task.repo}): {e}")
            return (task, None, e)

    with ThreadPoolExecutor(max_workers=1) as executor:
        futures = {
            executor.submit(publish_one, repo_key, repo_root): repo_key for repo_key, repo_root in repo_roots.items()
        }
        for future in as_completed(futures):
            task, dashboard_collection, exc = future.result()
            if exc is not None or dashboard_collection is None:
                failures.append(task)
            else:
                dashboards[task] = dashboard_collection

    if failures:
        print("\nSome repos failed to publish:", ", ".join(str(t) for t in failures))

OK: /tmp/asv-publish-lvs72acr/Textualize/rich


summaries: 100%|██████████| 32/32 [00:00<00:00, 355.97it/s]


Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/Textualize_rich.fc.pkl
OK: /tmp/asv-publish-lvs72acr/nilearn/nilearn


summaries: 100%|██████████| 14/14 [00:00<00:00, 249.90it/s]


Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/nilearn_nilearn.fc.pkl
OK: /tmp/asv-publish-lvs72acr/scikit-image/scikit-image


...

Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/tskit-dev_msprime.fc.pkl



/home/asehgal/formulacode/datasmith/src/datasmith/scrape/scrape_dashboards.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_summaries_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
09:36:21 WARNING  root: Couldn't find 784f4d19 in branches (HEAD)


OK: /tmp/asv-publish-lvs72acr/Unidata/MetPy


summaries: 100%|██████████| 128/128 [00:00<00:00, 353.10it/s]


Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/Unidata_MetPy.fc.pkl


In [8]:
dashboards = {}
# dashboard_dir
for db_pth in dashboard_dir.glob("*.fc.pkl"):
    owner, repo = db_pth.stem.split("_", 1)
    task = Task(owner=owner, repo=repo, sha=None, tag=None)
    if task in dashboards:
        continue
    dashboards[task] = BenchmarkCollection.load(db_pth)
len(dashboards)

63

In [7]:
dashboards

{Task(owner='scverse', repo='spatialdata.fc', sha=None, commit_date=0.0, tag=None): BenchmarkCollection(base_url='/tmp/asv-publish-v_37jmqo/scverse/spatialdata/html', collected_at=datetime.datetime(2025, 9, 7, 1, 11, 37, 608880, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2025, 9, 7, 1, 11, 37, 610334, tzinfo=datetime.timezone.utc), param_keys=['branch', 'machine', 'num_cpu', 'python']),
 Task(owner='tqdm', repo='tqdm.fc', sha=None, commit_date=0.0, tag=None): BenchmarkCollection(base_url='/tmp/asv-publish-v_37jmqo/tqdm/tqdm/html', collected_at=datetime.datetime(2025, 9, 7, 1, 2, 28, 449413, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2025, 9, 7, 1, 2, 28, 450818, tzinfo=datetime.timezone.utc), param_keys=['alive-progress', 'branch', 'machine', 'num_cpu', 'progressbar2', 'python', 'rich']),
 Task(owner='DASDAE', repo='dascore.fc', sha=None, commit_date=0.0, tag=None): BenchmarkCollection(base_url='/tmp/asv-publish-v_37jmqo/DASDAE/dascore/html', collect